In [1]:
import torch
from Detanet.detanet_model import nn_vib_analysis,Lorenz_broadening,DetaNet,get_raman_intensity,uv_model,nmr_calculator,nmr_sca,charge_model
import numpy as np
device=torch.device('cuda')
dtype=torch.float32

In [2]:
import json
import torch
import re

# 提取pos和z的函数
def extract_pos_z(output):
    try:
        # 使用正则表达式从文本中提取pos和z
        pos_match = re.search(r'pos tensor: (\[\[.*?\]\])', output)
        z_match = re.search(r'z tensor: (\[.*?\])', output)

        # 提取pos和z的列表
        pos_tensor = eval(pos_match.group(1)) if pos_match else []
        z_tensor = eval(z_match.group(1)) if z_match else []

        # 将pos tensor中的字符串数字转换为float类型
        for i, item in enumerate(pos_tensor):
            # 检查是否可以转换为浮点数
            pos_tensor[i] = [float(coord) for coord in item if re.match(r"^[+-]?\d*\.?\d+(e[+-]?\d+)?$", coord)]

        # 转为torch.tensor
        pos_tensor = torch.tensor(pos_tensor, dtype=torch.float32) if pos_tensor else torch.tensor([], dtype=torch.float32)
        z_tensor = torch.LongTensor(z_tensor) if z_tensor else torch.LongTensor([])
        print(pos_tensor)
        return pos_tensor, z_tensor
    except Exception as e:
        print(f"Error processing output: {output}. Error: {e}")
        return torch.tensor([], dtype=torch.float32), torch.LongTensor([])

# 读取JSON文件
def process_json_file(json_file_path,str):
    # 打开并加载 JSON 文件
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # 遍历 JSON 数据，提取所有的 pos 和 z
    results = []
    for entry in data:
        output = entry.get(str, "") 
        pos, z = extract_pos_z(output)
        results.append({"pos": pos, "z": z})
    
    return results

json_file_path = "generate_output(1.19).json" 

# 处理 JSON 文件
pre_results = process_json_file(json_file_path,"Generated Output")
act_results= process_json_file(json_file_path,"Actual Output")

print(len(pre_results))
print(len(act_results))

tensor([[ 1.7990e-02, -1.1564e+00, -4.7930e-03],
        [ 1.8490e-03, -3.1390e-03,  2.6790e-03],
        [-1.6535e-02,  1.3719e+00,  9.5490e-03],
        [-3.2072e-02,  2.5252e+00,  1.5650e-02]])
tensor([[ 0.6071,  0.8594, -0.6558],
        [-0.0348, -0.1750,  0.0555],
        [ 0.0990, -0.2324,  1.6049],
        [-1.3788, -0.6160,  1.6456],
        [-2.0302, -1.7080,  2.3651],
        [-2.2600, -0.3307,  2.7216],
        [-1.5836, -0.1506,  0.2054],
        [ 0.3379,  1.7070, -0.2828],
        [ 0.3064, -1.0995, -0.4087],
        [ 0.8301, -0.9166,  2.0350],
        [ 0.2481,  0.7663,  2.0253],
        [-2.8806, -2.2168,  1.9190],
        [-1.4677, -2.2852,  3.0940],
        [-1.9661,  0.8738,  0.1837],
        [-2.1771, -0.7705, -0.4664]])
tensor([[-1.3896e-02,  1.3150e+00,  1.2798e-01],
        [ 1.1136e+00,  2.0162e+00,  2.2200e-03],
        [ 1.1916e+00,  1.3896e+00, -1.1750e-01],
        [ 6.8948e-02,  8.3039e-02, -1.1224e-01],
        [-1.1535e+00, -6.1508e-01, -1.4201e-01],
  

In [3]:
'''Loading model'''
vib_model=nn_vib_analysis(device=device,Linear=False,scale=0.965)
nmr_model=nmr_calculator(device=device)
uv_model_=uv_model(device=device)

c:\Users\Raytine\.conda\envs\pytorch\Lib\site-packages\torch\jit\_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [4]:
act_results[2]['z']

tensor([7, 6, 7, 6, 7, 6, 7, 1, 1, 1, 1])

In [5]:
pre_results[2]['z']

tensor([7, 6, 7, 6, 7, 6, 7, 1, 1, 1, 1, 1])

In [6]:
import torch
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# Initialize lists to store valid results
r2_nir = []
rmse_nir = []
mae_nir = []
r2_raman = []
rmse_raman = []
mae_raman = []
r2_uv = []
rmse_uv = []
mae_uv = []
x_axis = torch.linspace(500, 4000, 3501)

# Initialize variables to accumulate sum of valid metrics
total_r2_nir = 0
total_rmse_nir = 0
total_mae_nir = 0
total_r2_raman = 0
total_rmse_raman = 0
total_mae_raman = 0
total_r2_uv = 0
total_rmse_uv = 0
total_mae_uv = 0

valid_count_nir = 0  # Counter for valid NIR data points
valid_count_raman = 0  # Counter for valid Raman data points
valid_count_uv = 0 
# Loop through the results and compute metrics
for i in range(len(pre_results)):
    try:
        # Get frequency, iir, araman from the model predictions
        freq, iir, araman = vib_model(z=act_results[i]["z"], pos=act_results[i]["pos"])
        freq_pre, iir_pre, araman_pre = vib_model(z=pre_results[i]["z"], pos=pre_results[i]["pos"])
        
        # Replace NaN values in freq with 0
        freq = torch.nan_to_num(freq, nan=0.0)
        freq_pre = torch.nan_to_num(freq_pre, nan=0.0)
        
        freq = freq.to('cpu')
        freq_pre = freq_pre.to('cpu')

        iir = iir.to('cpu')
        iir_pre = iir_pre.to('cpu')

        araman = araman.to('cpu')
        araman_pre = araman_pre.to('cpu')

        # Apply Lorentzian broadening
        yir = Lorenz_broadening(freq, iir, c=x_axis, sigma=15).detach().numpy()
        yir_pre = Lorenz_broadening(freq_pre, iir_pre, c=x_axis, sigma=15).detach().numpy()

        yraman_act = Lorenz_broadening(freq, araman, c=x_axis, sigma=12)
        yraman_act_pre = Lorenz_broadening(freq_pre, araman_pre, c=x_axis, sigma=12)

        # Get Raman intensity
        yraman = get_raman_intensity(x_axis, yraman_act).detach().numpy()
        yraman_pre = get_raman_intensity(x_axis, yraman_act_pre).detach().numpy()

        # Compute MSE, RMSE, and R2 for NIR and Raman
        mse_nir_value = mean_squared_error(yir, yir_pre)
        rmse_nir_value = np.sqrt(mse_nir_value)
        mae_nir_value = mean_absolute_error(yir, yir_pre)
        r2_nir_value = r2_score(yir, yir_pre)
        
        mse_raman_value = mean_squared_error(yraman, yraman_pre)
        rmse_raman_value = np.sqrt(mse_raman_value)
        mae_raman_value = mean_absolute_error(yraman, yraman_pre)
        r2_raman_value = r2_score(yraman, yraman_pre)


        # If values are invalid, mark as irrelevant (set to 0)
        if mse_nir_value > 1 or r2_nir_value < 0:
            r2_nir_value = 0
            rmse_nir_value = 0
            mae_nir_value = 0

        if mse_raman_value > 1 or r2_raman_value < 0:
            r2_raman_value = 0
            rmse_raman_value = 0
            mae_raman_value = 0
        

        # Append valid or marked-as-invalid results
        r2_nir.append(r2_nir_value)
        rmse_nir.append(rmse_nir_value)
        mae_nir.append(mae_nir_value)
        r2_raman.append(r2_raman_value)
        rmse_raman.append(rmse_raman_value)
        mae_raman.append(mae_raman_value)

        # Accumulate for averaging
        total_r2_nir += r2_nir_value
        total_rmse_nir += rmse_nir_value
        total_mae_nir += mae_nir_value
        total_r2_raman += r2_raman_value
        total_rmse_raman += rmse_raman_value
        total_mae_raman += mae_raman_value

        # Count the valid or invalid data points
        valid_count_nir += 1
        valid_count_raman += 1
    except Exception as e:
        print(f"Skipping index {i} due to error: {e}")

# Calculate the average metrics for all data (including marked invalid)
if valid_count_nir > 0:
    average_r2_nir = total_r2_nir / valid_count_nir
    average_rmse_nir = total_rmse_nir / valid_count_nir
    average_mae_nir = total_mae_nir / valid_count_nir
    print(f"Average R² for NIR: {average_r2_nir}")
    print(f"Average RMSE for NIR: {average_rmse_nir}")
    print(f"Average MAE for NIR: {average_mae_nir}")
else:
    print("No valid NIR results to calculate averages.")

if valid_count_raman > 0:
    average_r2_raman = total_r2_raman / valid_count_raman
    average_rmse_raman = total_rmse_raman / valid_count_raman
    average_mae_raman = total_mae_raman / valid_count_raman
    print(f"Average R² for Raman: {average_r2_raman}")
    print(f"Average RMSE for Raman: {average_rmse_raman}")
    print(f"Average MAE for Raman: {average_mae_raman}")
else:
    print("No valid Raman results to calculate averages.")


c:\Users\Raytine\.conda\envs\pytorch\Lib\site-packages\torch\jit\_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "
c:\Users\Raytine\.conda\envs\pytorch\Lib\site-packages\torch\jit\_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "
c:\Users\Raytine\.conda\envs\pytorch\Lib\site-packages\torch\jit\_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class

Average R² for NIR: 0.9399999956405342
Average RMSE for NIR: 3.890655162106946e-05
Average MAE for NIR: 8.405071725707259e-06
Average R² for Raman: 0.9399999907964417
Average RMSE for Raman: 0.1351461498525182
Average MAE for Raman: 0.026148305361417813


In [11]:
import torch
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# Initialize lists to store valid results

r2_uv = []
rmse_uv = []
mae_uv = []
x_axis = torch.linspace(500, 4000, 3501)

# Initialize variables to accumulate sum of valid metrics

total_r2_uv = 0
total_rmse_uv = 0
total_mae_uv = 0


valid_count_uv = 0 
# Loop through the results and compute metrics
for i in range(len(pre_results)):
    try:
  
        uv=uv_model_(z=act_results[i]["z"].to('cuda'), pos=act_results[i]["pos"].to('cuda')).detach().cpu().numpy()
        uv_pre=uv_model_(z=pre_results[i]["z"].to('cuda'), pos=pre_results[i]["pos"].to('cuda')).detach().cpu().numpy()

        mse_uv_value = mean_squared_error(uv, uv_pre)
        rmse_uv_value = np.sqrt(mse_uv_value)
        mae_uv_value = mean_absolute_error(uv, uv_pre)
        r2_uv_value = r2_score(uv, uv_pre)

        
        if mse_uv_value > 1 or r2_uv_value < 0:
            r2_uv_value = 0
            rmse_uv_value = 0
            mae_uuv_value = 0

        r2_uv.append(r2_uv_value)
        rmse_uv.append(rmse_uv_value)
        mae_uv.append(mae_uv_value)

        total_r2_uv += r2_uv_value
        total_rmse_uv += rmse_uv_value
        total_mae_uv += mae_uv_value

        valid_count_uv += 1

    except Exception as e:
        print(f"Skipping index {i} due to error: {e}")

if valid_count_uv > 0:
    average_r2_uv = total_r2_uv / valid_count_uv
    average_rmse_uv = total_rmse_uv / valid_count_uv
    average_mae_uv = total_mae_uv / valid_count_uv
    print(f"Average R² for uv: {average_r2_uv}")
    print(f"Average RMSE for uv: {average_rmse_uv}")
    print(f"Average MAE for uv: {average_mae_uv}")
else:
    print("No valid uv results to calculate averages.")

Average R² for uv: 0.949919558029666
Average RMSE for uv: 0.000164344891982921
Average MAE for uv: 0.0006012593990845955
